In [ ]:
import json
from http.client import RemoteDisconnected
from json import JSONDecodeError
from pathlib import Path
from typing import Optional, TypeAlias
from urllib.error import HTTPError
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
from monty.json import MontyEncoder
from mp_api.client import MPRester
from numpy.typing import ArrayLike
from tqdm import tqdm

In [ ]:
SERVER = "https://aflow.org"
API = "/API/aflux/?"
MP_API_KEY = "NBjBZuzKiGCZIBrIyLTYfwO35iHMiWwX"
ROOT = ".."

In [ ]:
AfluxResponse: TypeAlias = list[dict[str, str]] | list
OptionalRange: TypeAlias = Optional[tuple[int, int]]
PathLike: TypeAlias = str | Path

In [ ]:
def aflux_request(
    matchbook: str,
    paging: Optional[int] = None,
    paging_range: OptionalRange = None,
    no_directives: bool = False,
    retries: int = 0,
) -> AfluxResponse:
    """Download a AFLUX response and return it as list of dictionaries"""
    if paging is not None and paging_range is not None:
        raise ValueError("Cannot specify both paging and paging_range")

    request_url = SERVER + API + matchbook

    if not no_directives:
        if paging is not None:
            request_url += f",$paging({paging}),format(json)"
        elif paging_range is not None:
            request_url += f",$paging({paging_range[0]},{paging_range[1]}),format(json)"
        else:
            request_url += ",$paging(0),format(json)"

    try:
        server_response = urlopen(request_url)
    except RemoteDisconnected:
        if retries > 0:
            print("RemoteDisconnected error, retrying...")
            return aflux_request(matchbook, paging, paging_range, no_directives, retries - 1)
        else:
            print("RemoteDisconnected error, no more retries left")
    else:
        response_content = server_response.read().decode("utf-8")

    # Basic error handling
    if server_response.getcode() == 200:
        try:
            print("AFLUX request successful!")
            return json.loads(response_content)
        except JSONDecodeError:
            pass

    print("AFLUX request failed!")
    print(f"  URL: {request_url}")
    print(f"  Response: {response_content}")
    return None

In [ ]:
def aflux_help(keyword: Optional[str] = None) -> None:
    """Print the build in help of AFLUX"""
    if keyword is None:
        # General help (https://aflow.org/API/aflux/?)
        help_data = aflux_request("", no_directives=True)
        print("\n".join(help_data))
    else:
        # Help regarding a specific keyword (https://aflow.org/API/aflux/?help(keyword))
        help_data = aflux_request(f"help({keyword})")
        for key, entry in help_data.items():
            print(key)
            print(f"  description: {entry['description']}")
            print(f"  units: {entry['units']}")
            print(f"  status: {entry['status']}")
            comment = "\n    ".join(entry["__comment__"]).strip()
            if comment:
                print(f"  comment:\n    {comment}")

In [ ]:
def aflux_get_contcar(entry: dict[str, str]) -> str | None:
    """Get a CONTCAR from AFLUX"""
    request_url = "http://" + entry["aurl"].replace(":", "/") + "/" + "CONTCAR.relax"
    server_response = urlopen(request_url)
    response_content = server_response.read().decode("utf-8")
    if server_response.getcode() == 200:
        # Fix POSTCAR if in VASP4 format
        poscar_lines = response_content.split("\n")
        # Add species names if missing
        if poscar_lines[5].strip()[0].isnumeric():
            poscar_lines.insert(5, " ".join(entry["species"]))
        poscar = "\n".join(poscar_lines)
        return poscar
    print("AFLUX request failed!")
    print(f"  URL: {request_url}")
    print(f"  Response: {response_content}")
    return None

In [ ]:
def aflux_get_property(entry: dict[str, str], property: str) -> float | None:
    """Get a property from AFLUX"""
    aurl = entry["aurl"].replace(":", "/")
    request_url = f"http://{aurl}/?{property}"
    server_response = urlopen(request_url)
    response_content = server_response.read().decode("utf-8")
    if server_response.getcode() == 200:
        # Fix POSTCAR if in VASP4 format
        poscar_lines = response_content.split("\n")
        # Add species names if missing
        if poscar_lines[5].strip()[0].isnumeric():
            poscar_lines.insert(5, " ".join(entry["species"]))
        poscar = "\n".join(poscar_lines)
        return poscar
    print("AFLUX request failed!")
    print(f"  URL: {request_url}")
    print(f"  Response: {response_content}")
    return None

In [ ]:
def clean_nones(value):
    """
    Recursively remove all None values from dictionaries and lists, and returns
    the result as a new dictionary or list.
    """
    if isinstance(value, list):
        return [clean_nones(x) for x in value if x is not None]
    elif isinstance(value, dict):
        return {key: clean_nones(val) for key, val in value.items() if val is not None}
    else:
        return value

In [ ]:
def clean_mp_entries(entries):
    """
    Clean a list of entries from the MP database, removing all None values.
    """
    for entry in entries:
        if entry["deprecated"] or entry["warnings"]:
            del entry
            continue

        entry["structure"] = entry["structure"].to("POSCAR")
        entry["spacegroup"] = entry["symmetry"]["number"]

        entry["symmetry"] = None
        entry["deprecated"] = None
        entry["warnings"] = None

    return clean_nones(entries)

In [ ]:
def load_sg_from_dataset(path: PathLike) -> list[int]:
    if not isinstance(path, Path):
        path = Path(path)

    dataset = []
    for i in range(1, 231):
        file = path / f"data_{i}.json"
        if not file.exists():
            continue

        data = json.loads(file.read_text())
        if data:
            if "spacegroup_relax" in data[0].keys():
                for entry in data:
                    entry["spacegroup"] = entry["spacegroup_relax"]
                    del entry["spacegroup_relax"]

            dataset += [int(entry["spacegroup"]) for entry in data]

    return dataset

In [ ]:
def plot_spacegroup_distribution(dataset: ArrayLike, ax=None, title=None) -> None:
    sg_unique, sg_count = np.unique(dataset, return_counts=True)
    sg_percentage = sg_count / np.sum(sg_count)

    for i in range(1, 231):
        if i not in sg_unique:
            sg_unique = np.insert(sg_unique, i, i)
            sg_percentage = np.insert(sg_percentage, i, 0)

    print(f"Spacegroup without data: {np.where(sg_percentage == 0)[0]}")

    if ax is None:
        ax = plt.gca()

    ax.bar(sg_unique, sg_percentage, log=True, width=1)

    ax.set_xlim(1, 230)
    ax.set_xlabel("Space group")
    ax.set_ylabel("Percentage of entries")
    if title:
        ax.set_title(title)

In [ ]:
%%script false --no-raise-error
data = aflux_request(f"spacegroup_relax(123)", paging_range=(4,200000))
print(data)
# Path(f"{ROOT}/dataset/aflow/data_123_3.json").write_text(json.dumps(data, indent=4))

In [ ]:
# %%script false --no-raise-error
for sg in tqdm(range(1, 231)):
    file = Path(f"{ROOT}/dataset/aflow/data_{sg}.json")

    if file.is_file():
        continue

    i = 1
    data = None
    total_data = []
    while data != []:
        try:
            data = aflux_request(f"spacegroup_relax({sg})", paging_range=(i, 200_000))
        except HTTPError as e:
            print(f"Unable to fetch data for spacegroup {sg}: {e}\n")
        else:
            i += 1
            total_data += data

    if total_data:
        file.write_text(json.dumps(total_data, indent=4))

In [ ]:
for i in tqdm(range(1, 231)):
    file = Path(f"{ROOT}/dataset/mp/data_{i}.json")
    if file.is_file():
        continue

    with MPRester(MP_API_KEY, mute_progress_bars=True, use_document_model=False) as mpr:
        docs = mpr.materials.summary.search(
            spacegroup_number=i,
            fields=["material_id", "symmetry", "structure", "deprecated", "warnings"],
        )

    docs = clean_mp_entries(docs)

    file.write_text(json.dumps(docs, indent=4, allow_nan=False, cls=MontyEncoder))

In [ ]:
aflow_data = load_sg_from_dataset("dataset/aflow")
print(f"Number of entries (aflow): {len(aflow_data)}")

mp_data = load_sg_from_dataset("dataset/mp")
print(f"Number of entries (mp): {len(mp_data)}")

total_data = aflow_data + mp_data
print(f"Number of entries (total): {len(total_data)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, dataset in enumerate([aflow_data, mp_data, total_data]):
    plot_spacegroup_distribution(dataset, axes[i], title=["Aflow", "MP", "Total"][i] + " dataset")

plt.show()